In [2]:
!pip install wordcloud matplotlib pymupdf sentence-transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 65.7 MB/s eta 0:00:00


In [23]:
import fitz
import re
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
from collections import Counter
from sentence_transformers import SentenceTransformer, util
from PIL import Image
from sentence_transformers import SentenceTransformer
import numpy as np
import gradio as gr

def extract_text_from_pdf(pdfPath: str) -> str:
    try:
      doc = fitz.open(pdfPath)
    except Exception:
      return ""

    if doc.page_count == 0:
      doc.close()
      return ""

    text = ""
    for page in doc:
      text += page.get_text()

    doc.close()

    return text

def clear_text(pdf_text: str) -> str:
  if pdf_text == "":
    return ""

  pdf_text = pdf_text.lower()
  pdf_text = re.sub(r"http\S+|www\.\S+", " ", pdf_text)
  pdf_text = pdf_text.replace("\n", " ")
  pdf_text = pdf_text.replace("\t", " ")
  pdf_text = pdf_text.replace("\r", " ")

  return pdf_text

def generate_wordcount(pdf_text: str) -> Counter:
  if pdf_text == "":
    return Counter()

  final_worlds = []
  words = pdf_text.split()
  stopwords = set(STOPWORDS)

  for word in words:
    if word not in stopwords:
      final_worlds.append(word)

  return Counter(final_worlds)

def generate_wordcloud_image(counts: Counter) -> Image.Image:
    wc = WordCloud(width=900, height=450, background_color="white")
    wc = wc.generate_from_frequencies(dict(counts))
    return Image.fromarray(wc.to_array())


def semantic_similarity(text_a: str, text_b: str) -> float:
    if not text_a or not text_b:
        return 0.0

   # limit size for speed (big PDFs can be huge)
    text_a = text_a[:15000]
    text_b = text_b[:15000]

    emb = _embedder.encode([text_a, text_b], normalize_embeddings=True)

    # dot product of normalized vectors == cosine similarity
    return float(np.dot(emb[0], emb[1]))

_embedder = SentenceTransformer("all-MiniLM-L6-v2")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
def compare(pdf_a, pdf_b):
    # Gradio gives file objects; .name is the saved temp path
    path_a = pdf_a.name if pdf_a else ""
    path_b = pdf_b.name if pdf_b else ""

    raw_a = extract_text_from_pdf(path_a)
    raw_b = extract_text_from_pdf(path_b)

    text_a = clear_text(raw_a)
    text_b = clear_text(raw_b)

    counts_a = generate_wordcount(text_a)
    counts_b = generate_wordcount(text_b)

    img_a = generate_wordcloud_image(counts_a)
    img_b = generate_wordcloud_image(counts_b)

    sem = semantic_similarity(text_a, text_b)

    same_topic = sem >= 0.65
    verdict = "✅ Same topic / concept" if same_topic else "❌ Different topic / concept"

    return img_a, img_b, verdict


In [25]:
demo = gr.Interface(
    fn=compare,
    inputs=[
        gr.File(label="Upload PDF A", file_types=[".pdf"], type="filepath"),
        gr.File(label="Upload PDF B", file_types=[".pdf"], type="filepath"),
    ],
    outputs=[
        gr.Image(label="WordCloud - PDF A"),
        gr.Image(label="WordCloud - PDF B"),
        gr.Textbox(label="Verdict"),
    ],
    title="PDF Topic Comparison",
    description="Upload two PDFs. See word clouds and whether they match by topic."
)


In [26]:
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://65334935dd99025069.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
